In [1]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


E:\Github\Machine-Learning-Projects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/04/11 20:34:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/04/11 20:34:11 INFO mlflow.store.db.utils: Updating database tables


2026/04/11 20:34:13 INFO mlflow.tracking.fluent: Experiment with name '.' does not exist. Creating a new experiment.


MLflow experiment: .


# Cotton Disease Prediction

Modern Image Classification Pipeline (April 2026)

Primary : DINOv2 ViT-S/14 backbone (frozen head-only, then full fine-tune).
Alternative: ConvNeXt V2 Tiny (fine-tuning baseline via timm).
Timing  : Wall-clock per model.
Export  : metrics.json with per-model accuracy + timing; confusion matrix plot.
Data    : Auto-downloaded at runtime.


## Environment Setup


In [2]:
import os
# Define __file__ so save-path logic in the pipeline works correctly
__file__ = os.path.abspath('pipeline.py')
%matplotlib inline


## Dataset Setup

Datasets are **auto-downloaded** at runtime via the Kaggle API, sklearn, yfinance, or HuggingFace.

**Kaggle setup** (one-time): place your `kaggle.json` in `~/.kaggle/`.


In [3]:
# Kaggle API setup — credentials are read from ~/.kaggle/kaggle.json
# To create: https://www.kaggle.com/settings → API → Create New Token
import os, subprocess, sys
try:
    import kaggle  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
# Create project data & output directories
os.makedirs("data", exist_ok=True)
os.makedirs(os.path.join("outputs", "eda"), exist_ok=True)
os.makedirs(os.path.join("outputs", "models"), exist_ok=True)
os.makedirs(os.path.join("outputs", "results"), exist_ok=True)
print("Dataset directories ready")


Dataset directories ready


## Imports & Configuration


In [4]:
import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

IMG_SIZE, BATCH_SIZE, EPOCHS, LR = 224, 64, 3, 1e-4
SAVE_DIR = os.path.dirname(os.path.abspath(__file__))


## Get Transforms


In [5]:
def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE), transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


## Train Model


In [6]:
def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    from torchvision import datasets as tv_datasets
    train_ds = tv_datasets.CIFAR10(root="./data", train=True, download=True, transform=get_transforms(True))
    n_classes = 10

    # Cap training to 5K to prevent OOM / timeout with heavy backbones
    MAX_TRAIN = 5000
    if len(train_ds) > MAX_TRAIN:
        train_ds, _ = random_split(train_ds, [MAX_TRAIN, len(train_ds) - MAX_TRAIN])
    val_size = max(1, int(0.2 * len(train_ds)))
    train_sub, val_sub = random_split(train_ds, [len(train_ds) - val_size, val_size])
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, num_workers=0)
    metrics = {}

    # ── DINOv2 (primary) ──
    print()
    print("-- DINOv2 ViT-S/14 --")
    backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", pretrained=True)
    embed_dim = 384  # ViT-S/14

    class Classifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 256),
                                      nn.GELU(), nn.Dropout(0.3), nn.Linear(256, n_classes))
            for p in self.backbone.parameters(): p.requires_grad = False
        def forward(self, x):
            feat = self.backbone(x)
            return self.head(feat)

    model = Classifier().to(device)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.head.parameters(), lr=LR, weight_decay=0.01)

    best_acc = 0
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train(); total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            loss = criterion(model(imgs), labels); loss.backward()
            opt.step(); opt.zero_grad(); total_loss += loss.item()
        model.eval(); preds, gts = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                preds.extend(torch.argmax(model(imgs.to(device)), dim=-1).cpu().numpy())
                gts.extend(labels.numpy())
        val_acc = accuracy_score(gts, preds)
        print(f"  Epoch {epoch+1}/{EPOCHS} -- Loss: {total_loss/len(train_loader):.4f} -- Val Acc: {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            best_preds, best_gts = preds, gts
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))
    dino_elapsed = round(time.perf_counter() - t0, 1)

    print(f"  DINOv2 Best Val Accuracy: {best_acc:.4f} ({dino_elapsed}s)")
    print(classification_report(best_gts, best_preds, zero_division=0))
    metrics["DINOv2"] = {"val_accuracy": round(best_acc, 4), "epochs": EPOCHS, "time_s": dino_elapsed}

    # Confusion matrix
    cm = confusion_matrix(best_gts, best_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("DINOv2 Confusion Matrix")
    fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100, bbox_inches="tight")
    plt.close(fig)

    # ── ConvNeXt V2 (alternative baseline) ──
    print()
    print("-- ConvNeXt V2 Tiny --")
    try:
        import timm
        t1 = time.perf_counter()
        convnext = timm.create_model("convnextv2_tiny.fcmae_ft_in22k_in1k", pretrained=True, num_classes=n_classes).to(device)
        for _name, _param in convnext.named_parameters():
            if "head" not in _name:
                _param.requires_grad = False
        convnext_opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, convnext.parameters()), lr=LR * 0.5, weight_decay=0.01)
        for epoch in range(2):
            convnext.train(); total_loss = 0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                loss = criterion(convnext(imgs), labels); loss.backward()
                convnext_opt.step(); convnext_opt.zero_grad(); total_loss += loss.item()
        convnext.eval(); cv_preds, cv_gts = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                cv_preds.extend(torch.argmax(convnext(imgs.to(device)), dim=-1).cpu().numpy())
                cv_gts.extend(labels.numpy())
        cv_acc = accuracy_score(cv_gts, cv_preds)
        cv_elapsed = round(time.perf_counter() - t1, 1)
        print(f"  ConvNeXt V2 Val Accuracy: {cv_acc:.4f} ({cv_elapsed}s)")
        metrics["ConvNeXtV2"] = {"val_accuracy": round(cv_acc, 4), "epochs": 2, "time_s": cv_elapsed}
    except Exception as e:
        print(f"  ConvNeXt V2: {e}")

    return metrics


## Run Eda

Dataset statistics for image classification.


In [7]:
def run_eda(dataset, save_dir):
    """Dataset statistics for image classification."""
    print("\n" + "=" * 60)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    print(f"Total samples: {len(dataset)}")
    if hasattr(dataset, "classes"):
        print(f"Number of classes: {len(dataset.classes)}")
        from collections import Counter
        if hasattr(dataset, "targets"):
            class_counts = Counter(dataset.targets)
            print("\nSamples per class:")
            for cls_idx in sorted(class_counts.keys()):
                name = dataset.classes[cls_idx] if cls_idx < len(dataset.classes) else str(cls_idx)
                print(f"  {name}: {class_counts[cls_idx]}")
    elif hasattr(dataset, "class_to_idx"):
        print(f"Number of classes: {len(dataset.class_to_idx)}")
    print("EDA complete.")


## Main Pipeline


In [8]:
def main():
    print("=" * 60)
    print("IMAGE CLASSIFICATION | DINOv2 + ConvNeXt V2")
    print("=" * 60)
    # run_eda is called inside train_model() after dataset is loaded
    metrics = train_model()

    out_path = os.path.join(SAVE_DIR, "metrics.json")
    with open(out_path, "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"Metrics saved to {out_path}")


## Execute Pipeline

Run the complete ML pipeline end-to-end:


In [9]:
main()


IMAGE CLASSIFICATION | DINOv2 + ConvNeXt V2



-- DINOv2 ViT-S/14 --


Using cache found in C:\Users\ahmad/.cache\torch\hub\facebookresearch_dinov2_main


  Epoch 1/3 -- Loss: 2.0327 -- Val Acc: 0.6810


  Epoch 2/3 -- Loss: 1.4056 -- Val Acc: 0.7770


  Epoch 3/3 -- Loss: 0.9550 -- Val Acc: 0.8180


  DINOv2 Best Val Accuracy: 0.8180 (105.7s)
              precision    recall  f1-score   support

           0       0.78      0.94      0.85        95
           1       0.84      0.85      0.85       101
           2       0.75      0.67      0.71       100
           3       0.73      0.74      0.74        98
           4       0.74      0.84      0.78        91
           5       0.78      0.76      0.77        97
           6       0.85      0.92      0.88       110
           7       0.95      0.81      0.87       103
           8       0.92      0.83      0.87       110
           9       0.85      0.82      0.83        95

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000


-- ConvNeXt V2 Tiny --


  ConvNeXt V2 Val Accuracy: 0.5660 (73.2s)
Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Cotton Disease Prediction\metrics.json
